In [2]:
"""
Ce script réinsère les balises <supralinear> qui n'ont pas été intégrée par tei_transformer
"""
from pathlib import Path
from xml.etree import ElementTree
import re
from typing import TextIO
from lxml import etree
import shutil

missing_tag="supralinear"

def get_full_text(element):
    text = element.text or ""
    for child in element:
        print(f"texte:")
        if child.tag == missing_tag:
            break
        #text += get_full_text(child)  # Appel récursif pour les enfants
        elif child.tail:
            text += child.tail  # Inclure le texte après la balise enfant
    return text.strip()

def get_missing_tag(file_path: TextIO) -> list[dict]:
    """
    Cette fonction crée une liste de dictionnaire contenant:
    le chapitre (chap), le verset (verse), le contenu de la balise à insérer (content) et le texte qui précède l'endroit où la balise doit être insérée (text_before)
    """
    # Charger le fichier XML
    tree = ElementTree.parse(file_path)
    root = tree.getroot()

    # Liste pour stocker les résultats
    results = []

    # Variables pour suivre le contexte
    current_chap = None
    current_verse = None
    current_text = None

    # Parcourir les éléments du fichier XML
    for elem in root.iter():
        if elem.tag == "chap":
            current_chap = elem.text.strip() if elem.text else None
            #print(f'{current_chap}')
        elif elem.tag == "text":
            current_text = get_full_text(elem)  # Utiliser la fonction pour extraire tout le texte avant une balise supralinear
            #print(f'{current_text}')
        elif elem.tag == "verse_nb":
            current_verse = elem.text.strip() if elem.text else None
            #print(f'{current_verse}')
        elif elem.tag == missing_tag:
            content = elem.text.strip() if elem.text else None
            
            results.append({
                "chap": re.search(r'\d+', current_chap).group(0),
                "verse": current_verse,
                "content": content,
                "text_before": current_text            
            })
    # Afficher les résultats
    return results

In [3]:
def insert_supralinear_tags(results, in_file_path, out_file_path):
    #Faire une copie du fichier d'origine
    shutil.copy(in_file_path, out_file_path)
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(out_file_path, parser)
    root = tree.getroot()
    modified = False

    for entry in results:
        chap = entry["chap"]
        verse = entry["verse"]
        content = entry["content"]
        text_before = entry["text_before"]

        if not (chap and verse and content and text_before):
            continue

        # Trouver le <div type="chap" n=chap>
        chap_xpath = f".//div[@type='chap'][@n='{chap}']"
        chap_div = root.find(chap_xpath)
        if chap_div is None:
            continue

        # Trouver le <div type="verse" n=verse> à l'intérieur du chapitre
        verse_xpath = f".//div[@type='verse'][@n='{verse}']"
        verse_div = chap_div.find(verse_xpath)
        if verse_div is None:
            continue

        # Dernier mot de text_before
        last_word = text_before.strip().split()[-1]
        print(last_word)

        # Chercher le <w> dont le texte correspond au dernier mot
        for w in verse_div.iter("w"):
            if (w.text or "").strip() == last_word:
                # Créer le nouveau tag
                #new_w = etree.Element("w", reconstructed="0")
                #hi = etree.SubElement(new_w, "hi", rend="supralinear")
                hi = etree.SubElement(w, "hi", rend="supralinear")
                hi.text = content
                # Insérer juste après ce <w>
                #w.addnext(new_w)
                modified = True
                break

    if modified:
        #pprint.pprint(etree.tostring(root))
        tree.write(out_file_path, encoding="utf-8", pretty_print=True, xml_declaration=True)
        print(f"Modifications enregistrées dans {out_file_path}")
    else:
        print("Aucune modification effectuée.")

In [ ]:
if __name__ == "__main__":
    #Lecture de l'ensemble des fichier xml

        file_path = Path("../raw_files/ms_f.xml")
        results = get_missing_tag(file_path)
        for result in results:
            print(result)
        

In [12]:
        in_file_path = "../tei_files/" + file_path.name
        out_file_path = Path(in_file_path).with_name(file_path.stem + "_with_tag.xml")
        insert_supralinear_tags(results, in_file_path, out_file_path)

Modifications enregistrées dans ../tei_files/ms_f_with_tag.xml


In [27]:
"""
Supprime les espaces inutiles, et harmonise les quotes d'un fichier xml
"""
from lxml import etree
import pprint

parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse("../tei_files/ms_b.xml", parser)
root = tree.getroot()
xml = etree.tostring(root)
pprint.pprint(etree.tostring(root))
tree.write("../tei_files/ms_b_formated.xml", encoding="utf-8", pretty_print=True, xml_declaration=True)


(b'<root><ms name="Manuscript B"><div type="chap" n="10"><div type="verse" n="1'
 b'9cd"><line n="1" folio="t-s 12.871 recto"/><w reconstructed="0">&#1494;&#151'
 b'2;&#1506;</w><w reconstructed="0">&#1504;&#1511;&#1500;&#1492;</w><w reconst'
 b'ructed="0">&#1502;&#775;&#1492;&#775;</w><w reconstructed="0">&#1494;&#1512;'
 b'&#1506;</w><w reconstructed="0">&#1500;&#1488;&#1504;&#1493;&#1513;</w><stic'
 b'h/><w reconstructed="0">&#1494;&#1512;&#1506;</w><w reconstructed="0">&#1504'
 b';&#1511;&#1500;&#1492;</w><w reconstructed="0">&#1506;&#1493;&#1489;&#1512;<'
 b'/w><w reconstructed="0">&#1502;&#1510;&#1493;&#1492;&#1475;</w></div><div ty'
 b'pe="verse" n="20"><line n="2" folio="t-s 12.871 recto"/><w reconstructed="0"'
 b'>&#1489;&#1497;&#1503;</w><w reconstructed="0">&#1488;&#1495;&#1497;&#1501;<'
 b'/w><w reconstructed="0">&#1512;&#1488;&#1513;&#1501;</w><w reconstructed="0"'
 b'>&#1504;&#1499;&#1489;&#1491;</w><stich/><w reconstructed="0">&#1493;&#1497;'
 b'&#1512;&#1488;</w><w reco

In [46]:
"""
Insère une balise <lb\> avant chaque ligne qui n'est pas précédée d'une balise <div type="verse"> ou <lb>
"""

from lxml import etree

xml_path = "../tei_files/ms_b.xml"
out_path = "../tei_files/ms_b_with_lb.xml"

parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse(xml_path, parser)
root = tree.getroot()

n=0
for line in root.xpath('.//line'):
    parent = line.getparent()
    prev = line.getprevious()

    if prev != None:

        if prev.tag != "div" and prev.attrib.get("type")!= "verse" and prev.tag != "lb":
            print(f"{n}")
            n=n+1
            line.addprevious(etree.Element("lb"))

tree.write(out_path, encoding="utf-8", pretty_print=True, xml_declaration=True)
print(f"Fichier modifié enregistré sous {out_path}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
Fichier modifié enregistré sous ../tei_files/ms_b_with_lb.xml


<>:1: SyntaxWarning: invalid escape sequence '\>'
<>:1: SyntaxWarning: invalid escape sequence '\>'
/var/folders/4v/l62_h42s4rg1hyzc_kg9l0qm0000gn/T/ipykernel_89335/70680188.py:1: SyntaxWarning: invalid escape sequence '\>'
  """
